# Google Search Console Sentiment Analysis - Interactive Notebook

This notebook provides an interactive way to analyze your GSC data with sentiment analysis and entity extraction.

## Setup

Make sure you have:
1. Installed all requirements: `pip install -r requirements.txt`
2. Downloaded spaCy model: `python -m spacy download en_core_web_sm`
3. Set up Google Search Console API credentials (credentials.json)

In [ ]:
# Import required modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gsc_connector import GSCConnector
from sentiment_analyzer import SentimentAnalyzer
from entity_analyzer import EntityAnalyzer
from insights_generator import InsightsGenerator
from visualizations import create_visualizations

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Connect to Google Search Console

In [ ]:
# Initialize GSC connector
gsc = GSCConnector(credentials_path='credentials.json')
gsc.authenticate()

# List available sites
sites = gsc.list_sites()
print("Available sites:")
for i, site in enumerate(sites, 1):
    print(f"{i}. {site}")

## 2. Fetch Search Console Data

In [ ]:
# Set your site URL here
SITE_URL = 'https://www.yoursite.com'  # Change this!
DAYS = 30  # Number of days to analyze

# Fetch data
print(f"Fetching data for {SITE_URL} (last {DAYS} days)...")
gsc_data = gsc.get_recent_data(
    site_url=SITE_URL,
    days=DAYS,
    dimensions=['query', 'page']
)

print(f"Retrieved {len(gsc_data.get('rows', []))} queries")

## 3. Perform Sentiment Analysis

In [ ]:
# Initialize sentiment analyzer
sentiment_analyzer = SentimentAnalyzer()

# Analyze dataset
df = sentiment_analyzer.analyze_dataset(gsc_data)
print(f"Analyzed {len(df)} queries")

# Get summary
sentiment_summary = sentiment_analyzer.get_sentiment_summary(df)

# Display first few rows
df.head(10)

In [ ]:
# Display sentiment summary
print("\n=== SENTIMENT SUMMARY ===")
print(f"Total Queries: {sentiment_summary['total_queries']:,}")
print(f"Total Clicks: {sentiment_summary['total_clicks']:,}")
print(f"Total Impressions: {sentiment_summary['total_impressions']:,}")
print(f"Average CTR: {sentiment_summary['avg_ctr']*100:.2f}%")
print(f"Average Position: {sentiment_summary['avg_position']:.2f}")
print("\nSentiment Distribution:")
for sentiment, count in sentiment_summary['sentiment_distribution'].items():
    print(f"  {sentiment}: {count:,} ({count/sentiment_summary['total_queries']*100:.1f}%)")

## 4. Extract and Analyze Entities

In [ ]:
# Initialize entity analyzer
entity_analyzer = EntityAnalyzer()

# Extract entities
df = entity_analyzer.analyze_query_entities(df)
print(f"Extracted entities from {len(df)} queries")

# Get top entities
top_entities = entity_analyzer.get_top_entities(df, min_impressions=50)
print(f"\nTop 10 Entities:")
print(top_entities.head(10)[['entity', 'entity_type', 'clicks', 'impressions', 'position']])

In [ ]:
# Identify AI citation opportunities
entity_analysis = entity_analyzer.identify_citation_opportunities(df, min_clicks=20)

print(f"\n=== AI CITATION OPPORTUNITIES ===")
print(f"Total Entities: {entity_analysis['total_entities']}")
print(f"Citation-Ready Entities: {entity_analysis['citation_ready_entities']}")
print("\nTop Opportunities:")
for i, opp in enumerate(entity_analysis.get('top_citation_opportunities', [])[:5], 1):
    print(f"\n{i}. {opp['entity']} ({opp['entity_type']})")
    print(f"   Clicks: {opp['clicks']}, Position: {opp['avg_position']:.1f}")
    print(f"   Recommendation: {opp['recommendation']}")

## 5. Generate Insights

In [ ]:
# Initialize insights generator
insights_gen = InsightsGenerator()

# Analyze strengths and weaknesses
insights = insights_gen.analyze_strengths_weaknesses(df, sentiment_summary)

print("=== STRENGTHS ===")
for i, strength in enumerate(insights['strengths'][:5], 1):
    print(f"\n{i}. {strength['type']}")
    print(f"   {strength['description']}")
    print(f"   Impact: {strength['impact'].upper()}")

In [ ]:
print("=== WEAKNESSES ===")
for i, weakness in enumerate(insights['weaknesses'][:5], 1):
    print(f"\n{i}. {weakness['type']}")
    print(f"   {weakness['description']}")
    if 'opportunity' in weakness:
        print(f"   Fix: {weakness['opportunity']}")

In [ ]:
print("=== OPPORTUNITIES ===")
for i, opp in enumerate(insights['opportunities'][:5], 1):
    print(f"\n{i}. {opp['type']}")
    print(f"   {opp['description']}")
    print(f"   Recommendation: {opp['recommendation']}")
    if 'potential_clicks' in opp:
        print(f"   Potential: {opp['potential_clicks']:,} clicks")

## 6. AI Citation Strategy

In [ ]:
# Generate AI citation strategy
ai_strategy = insights_gen.generate_ai_citation_strategy(entity_analysis, df)

print("=== AI CITATION STRATEGY ===")
print("\nPriority Entities:")
for entity in ai_strategy.get('priority_entities', [])[:5]:
    print(f"  - {entity['entity']} ({entity['type']})")
    print(f"    Position: {entity['current_position']:.1f}, Traffic: {entity['traffic']} clicks")
    print(f"    Action: {entity['action']}")

print("\nContent Recommendations:")
for rec in ai_strategy.get('content_recommendations', []):
    print(f"  - {rec['type']} (Priority: {rec['priority']})")
    print(f"    {rec['description']}")
    print(f"    AI Impact: {rec['ai_impact']}")

## 7. Custom Analysis

Use this section for your own custom queries and analysis

In [ ]:
# Example: Find queries with high impressions but low CTR
low_ctr = df[(df['impressions'] > 1000) & (df['ctr'] < 0.02)].sort_values('impressions', ascending=False)
print("Queries with High Impressions but Low CTR:")
print(low_ctr[['query', 'impressions', 'clicks', 'ctr', 'position', 'sentiment']].head(10))

In [ ]:
# Example: Compare performance by sentiment
sentiment_perf = df.groupby('sentiment').agg({
    'clicks': 'sum',
    'impressions': 'sum',
    'ctr': 'mean',
    'position': 'mean',
    'query': 'count'
}).round(2)
sentiment_perf.columns = ['Total Clicks', 'Total Impressions', 'Avg CTR', 'Avg Position', 'Query Count']
print("\nPerformance by Sentiment:")
print(sentiment_perf)

In [ ]:
# Example: Find positive sentiment queries ranking poorly
positive_low_rank = df[(df['sentiment'] == 'positive') & (df['position'] > 20)].sort_values('impressions', ascending=False)
print("\nPositive Queries with Poor Rankings (Optimization Opportunities):")
print(positive_low_rank[['query', 'impressions', 'position', 'polarity']].head(10))

## 8. Export Results

In [ ]:
# Save the analyzed data
df.to_csv('notebook_analysis.csv', index=False)
print("Saved detailed analysis to notebook_analysis.csv")

# Generate full report
report = insights_gen.generate_summary_report(
    df, sentiment_summary, entity_analysis, insights
)
with open('notebook_report.txt', 'w') as f:
    f.write(report)
print("Saved report to notebook_report.txt")

print("\n" + report)